# PARC2026 Colab ブートストラップ

検証済みレシピ（`docs/env_setup.md` 参照、出典: [taku_sid氏のnote記事](https://note.com/taku_sid/n/n49a0008b29a6)）に基づく。
**ランタイム > ランタイムのタイプを変更 > GPU** を先に設定してから実行すること。

毎回のColabセッション開始時にこのノートブックの先頭セルから順に実行する。
ローカル(Claude Code / Codex CLI)で編集した自分のコード(`src/parc2026`)は別セルでimportする。

## 0. GPU確認

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Google Driveをマウント（モデル・アセットのキャッシュ永続化用）

Colabはセッションが切れるとローカルディスクが揮発し、モデルの再ダウンロードが発生する。
`HF_HOME` をDrive配下に向けてキャッシュを永続化する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/PARC2026'
os.makedirs(f'{DRIVE_ROOT}/hf_cache', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['MUJOCO_GL'] = 'egl'  # GPU描画に必須(評価実行前に毎回必要)

## 2. 自分のリポジトリをclone / pull（ローカルと連携）

`parc2026` はprivateリポジトリのため、Colabからcloneするには認証が必要。
ColabのSecrets（鍵アイコン）に `GH_TOKEN`（GitHubのPersonal Access Token, repo scope）を登録してから
下のセルを実行する。ローカルで編集→push→ここで `git pull` すれば常に最新コードで実行できる。

In [ ]:
from google.colab import userdata

GH_TOKEN = userdata.get('GH_TOKEN')
GITHUB_REPO_URL = f'https://{GH_TOKEN}@github.com/norikioka/parc2026.git'
REPO_DIR = '/content/parc2026'

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## 3. LeRobot本体をclone + pi/libero extrasでインストール

PyTorchはColabにプリインストール済みのCUDA対応版をそのまま使う（`--index-url` での入れ直しは基本不要。
バージョン不一致エラーが出た場合のみ `docs/env_setup.md` の手順2を参照して入れ直す）。

In [ ]:
LEROBOT_DIR = '/content/lerobot'
if not os.path.exists(LEROBOT_DIR):
    !git clone https://github.com/huggingface/lerobot.git {LEROBOT_DIR}

%cd {LEROBOT_DIR}
!pip install -q -e ".[pi,libero]"

## 4. HuggingFace認証（PaliGemmaはgated repoのため必須）

事前に https://huggingface.co で PaliGemma の利用規約に同意し、トークンを発行しておくこと。
Colabの場合は左メニューの鍵アイコン(Secrets)に `HF_TOKEN` を登録し、下のセルで読み込む方法が安全。

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## 5. 【重要】無印LIBEROの評価を先に完了させる

LIBERO-Plusをインストールすると素のLIBEROが**置き換わって同時運用できなくなる**。
無印LIBEROでの評価・提出物確認が終わるまでは、次のセル（LIBERO-Plus導入）を実行しないこと。

## 6. LIBERO-Plusへの切り替え（無印LIBEROでの検証が終わってから）

In [ ]:
LIBERO_PLUS_DIR = '/content/LIBERO-plus'
if not os.path.exists(LIBERO_PLUS_DIR):
    !git clone https://github.com/sylvestf/LIBERO-plus.git {LIBERO_PLUS_DIR}

%cd {LIBERO_PLUS_DIR}
!pip install -q --no-deps -e .
!pip install -q robosuite bddl easydict mujoco wand scikit-image gym

# アセット(オブジェクト・テクスチャ)は別配布。Sylvest/LIBERO-plusはHF上ではdatasetリポジトリなので
# --repo-type dataset を付けないと「Repository not found」になる(modelとして探しにいってしまうため)
!hf download Sylvest/LIBERO-plus assets.zip --repo-type dataset --local-dir /content/LIBERO-plus_assets
!unzip -q /content/LIBERO-plus_assets/assets.zip -d {LIBERO_PLUS_DIR}/libero/libero

## 7. 自分のコード(src/parc2026)をimport

In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/src')

import parc2026
print('parc2026 import OK')

## 8. 動作確認(ここから先はタスクごとのノートブック/スクリプトへ)

詰まった手順・エラーメッセージは `docs/env_setup.md` の「つまずきポイント一覧」に追記していくこと。